In [19]:
from pathlib import Path
import cv2
import numpy as np
import pickle

# =========================
# AJUSTES
# =========================
CAM_INDEX = 1
ROI_W = 0.55          # ancho ROI respecto al frame
ROI_H = 0.45          # alto ROI respecto al frame
MIN_AREA_FLOOR = 700  # minimo absoluto
MIN_AREA_RATIO = 0.006  # area minima relativa al ROI
BORDER_MARGIN = 1
BORDER_REJECT_AREA_RATIO = 0.98
USE_TEMPLATES = True  # usa pills/shape_templates.pkl si existe

PKL_PATH = Path("pills/shape_templates.pkl")


# =========================
# ROI central
# =========================
def get_center_roi(frame, roi_w=ROI_W, roi_h=ROI_H):
    h, w = frame.shape[:2]
    rw = int(w * roi_w)
    rh = int(h * roi_h)
    x1 = (w - rw) // 2
    y1 = (h - rh) // 2
    x2 = x1 + rw
    y2 = y1 + rh
    return (x1, y1, x2, y2), frame[y1:y2, x1:x2]


# =========================
# Blanco del papel desde esquinas de la ROI
# =========================
def estimate_paper_bgr(roi, frac=0.10):
    h, w = roi.shape[:2]
    ph = max(12, int(h * frac))
    pw = max(12, int(w * frac))

    patches = [
        roi[0:ph, 0:pw],
        roi[0:ph, w-pw:w],
        roi[h-ph:h, 0:pw],
        roi[h-ph:h, w-pw:w]
    ]

    pixels = np.concatenate([p.reshape(-1, 3) for p in patches], axis=0)
    return np.median(pixels, axis=0).astype(np.uint8)


# =========================
# Máscara usando diferencia contra el blanco del papel
# =========================
def build_mask_white_bg(roi):
    paper_bgr = estimate_paper_bgr(roi)

    lab = cv2.cvtColor(roi, cv2.COLOR_BGR2LAB).astype(np.float32)
    paper_lab = cv2.cvtColor(
        np.uint8([[paper_bgr]]), cv2.COLOR_BGR2LAB
    )[0, 0].astype(np.float32)

    # Distancia al color del papel
    delta = np.linalg.norm(lab - paper_lab, axis=2)
    delta = cv2.normalize(delta, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    delta = cv2.GaussianBlur(delta, (5, 5), 0)

    otsu_thr, _ = cv2.threshold(delta, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    # Umbral levemente permisivo para tonos claros
    mask = (delta > max(8, int(otsu_thr) - 10)).astype(np.uint8) * 255

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    kernel_big = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_big, iterations=2)

    return mask, paper_bgr, delta


# =========================
# Contornos válidos
# =========================
def get_valid_contours(mask, min_area=None):
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    h, w = mask.shape[:2]
    roi_area = float(h * w)
    if min_area is None:
        min_area = max(MIN_AREA_FLOOR, int(roi_area * MIN_AREA_RATIO))

    valid = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area:
            continue

        x, y, cw, ch = cv2.boundingRect(cnt)

        box_area = max(float(cw * ch), 1.0)
        fill_ratio = area / box_area
        aspect_box = max(cw, ch) / max(min(cw, ch), 1)

        # descarta tiras/fragmentos de borde o contornos muy pobres
        if fill_ratio < 0.28:
            continue
        if aspect_box > 6.0:
            continue

        # Solo descartar borde si parece ser region gigante/noise
        touches_border = (
            x <= BORDER_MARGIN
            or y <= BORDER_MARGIN
            or (x + cw) >= (w - BORDER_MARGIN)
            or (y + ch) >= (h - BORDER_MARGIN)
        )
        if touches_border and area > roi_area * BORDER_REJECT_AREA_RATIO and len(contours) > 1:
            continue

        valid.append(cnt)

    valid.sort(key=cv2.contourArea, reverse=True)
    return valid


def select_primary_contour(contours, roi_shape):
    if not contours:
        return None
    h, w = roi_shape[:2]
    cx0, cy0 = w / 2.0, h / 2.0

    # Si hay dos contornos grandes y alineados, unirlos (mitades de una pill bicolor)
    if len(contours) >= 2:
        c1, c2 = contours[0], contours[1]
        a1, a2 = cv2.contourArea(c1), cv2.contourArea(c2)
        if a1 > 1 and (a2 / a1) >= 0.45:
            x1, y1, w1, h1 = cv2.boundingRect(c1)
            x2, y2, w2, h2 = cv2.boundingRect(c2)
            c1x, c1y = x1 + w1 / 2.0, y1 + h1 / 2.0
            c2x, c2y = x2 + w2 / 2.0, y2 + h2 / 2.0

            y_aligned = abs(c1y - c2y) <= 0.35 * max(h1, h2)
            x_separated = abs(c1x - c2x) >= 0.25 * max(w1, w2)

            if y_aligned and x_separated:
                merged = cv2.convexHull(np.vstack([c1, c2]))
                return merged

    best = None
    best_score = -1e9
    for cnt in contours[:5]:
        area = cv2.contourArea(cnt)
        x, y, cw, ch = cv2.boundingRect(cnt)
        cx = x + cw / 2.0
        cy = y + ch / 2.0

        center_dist = np.hypot(cx - cx0, cy - cy0) / max(np.hypot(cx0, cy0), 1e-6)
        area_norm = area / max(float(w * h), 1.0)
        score = (2.0 * area_norm) - (0.75 * center_dist)
        if score > best_score:
            best_score = score
            best = cnt

    return best


def smooth_contour(cnt):
    # Suaviza bordes de caplets para estabilizar forma/color
    hull = cv2.convexHull(cnt)
    peri = cv2.arcLength(hull, True)
    eps = max(1.0, 0.008 * peri)
    approx = cv2.approxPolyDP(hull, eps, True)
    if len(approx) >= 5:
        return approx
    return hull


def get_external_edges(shape_hw, cnt):
    h, w = shape_hw
    edge = np.zeros((h, w), dtype=np.uint8)
    cv2.drawContours(edge, [cnt], -1, 255, 2)
    return edge


def get_internal_edges(roi, cnt):
    h, w = roi.shape[:2]
    obj = np.zeros((h, w), dtype=np.uint8)
    cv2.drawContours(obj, [cnt], -1, 255, -1)

    # Evita contar borde externo como borde interno
    inner = cv2.erode(obj, np.ones((7, 7), np.uint8), iterations=1)

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    edges = cv2.Canny(gray, 20, 80)
    int_edges = cv2.bitwise_and(edges, inner)
    int_edges = cv2.dilate(int_edges, np.ones((3, 3), np.uint8), iterations=1)

    pix = max(cv2.countNonZero(inner), 1)
    density = float(cv2.countNonZero(int_edges) / pix)
    return int_edges, density


# =========================
# Forma
# =========================
def classify_shape_rules(cnt):
    area = cv2.contourArea(cnt)
    peri = cv2.arcLength(cnt, True)
    if peri == 0:
        return "unknown", {}

    circularity = 4 * np.pi * area / (peri * peri)

    rect = cv2.minAreaRect(cnt)
    (_, _), (w, h), _ = rect
    w = max(w, 1e-6)
    h = max(h, 1e-6)
    aspect = max(w, h) / min(w, h)

    info = {
        "area": area,
        "peri": peri,
        "circularity": circularity,
        "aspect": aspect
    }

    # circulo casi perfecto
    if circularity >= 0.84 and aspect <= 1.12:
        return "circle", info

    # caplet redondeada (rango mas amplio)
    if 1.20 <= aspect < 3.60 and circularity >= 0.50:
        return "caplet", info

    # pill/capsule realmente alargada
    if aspect >= 3.60 or (aspect >= 2.30 and circularity < 0.50):
        return "pill", info

    if aspect > 1.12:
        return "caplet", info

    return "unknown", info


def load_templates():
    if USE_TEMPLATES and PKL_PATH.exists():
        with open(PKL_PATH, "rb") as f:
            return pickle.load(f)
    return None


def classify_shape(cnt, templates):
    if not templates:
        return classify_shape_rules(cnt)

    best_label = "unknown"
    best_score = float("inf")

    for label, tmpl in templates.items():
        score = cv2.matchShapes(cnt, tmpl, cv2.CONTOURS_MATCH_I1, 0.0)
        if score < best_score:
            best_score = score
            best_label = label

    label_rules, info = classify_shape_rules(cnt)
    info["match_score"] = best_score

    if best_score > 0.20:
        return label_rules, info

    return best_label, info


# =========================
# Color
# =========================
def _hsv_to_color_name(h, s, v):
    # Paleta objetivo (español): rosa, amarillo, azul, cafe, rojo, negro, naranja

    # negro (mas estrecho para no pisar azul oscuro)
    if v < 40:
        return "negro"
    if s < 16 and v < 95:
        return "negro"

    # azul (incluye azul claro/desaturado y azul oscuro util)
    if 82 <= h <= 132 and v >= 70 and s >= 14:
        return "azul"

    # rosa/magenta (prioridad alta para no caer en rojo)
    if (136 <= h <= 175 and v >= 85 and s >= 20) or (165 <= h <= 179 and v >= 95 and s <= 130):
        return "rosa"

    # amarillo oliva/saturado (evita confundir con cafe en tonos como #827922, #97922f)
    if 24 <= h <= 34 and s >= 150 and v >= 120:
        return "amarillo"

    # cafe: mas estricto para no pisar amarillos oliva
    if 10 <= h <= 24 and ((s >= 150 and v <= 125) or (s >= 105 and v <= 105)):
        return "cafe"

    # naranja (mas amplio hacia rojizo)
    if 6 <= h <= 21 and v >= 105 and s >= 45:
        return "naranja"

    # amarillo menos sensible (evita falsos en oliva oscuro)
    if 22 <= h <= 38 and v >= 120 and s >= 25 and not (s >= 150 and v <= 135):
        return "amarillo"

    # rojo
    if h <= 5 or h >= 176:
        return "rojo"

    # fallback sin crear etiquetas extra
    if 39 <= h <= 81:
        return "amarillo"
    if 6 <= h <= 21:
        return "naranja" if v >= 145 else "cafe"

    return "negro"

def estimate_size_label(cnt, roi_shape):
    h, w = roi_shape[:2]
    roi_area = max(float(h * w), 1.0)
    area = cv2.contourArea(cnt)
    ratio = area / roi_area

    # Cerca de camara -> ocupa mas area
    if ratio >= 0.130:
        return "grande", ratio
    if ratio >= 0.060:
        return "mediano", ratio
    return "pequeno", ratio

def classify_color(roi, cnt):
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    obj_mask = np.zeros(roi.shape[:2], dtype=np.uint8)
    cv2.drawContours(obj_mask, [cnt], -1, 255, -1)

    # medir color solo en el interior
    inner = cv2.erode(obj_mask, np.ones((5, 5), np.uint8), iterations=1)
    if cv2.countNonZero(inner) < 30:
        inner = obj_mask

    pixels = hsv[inner == 255]
    if len(pixels) == 0:
        return {
            "mode": "mono",
            "primary": "unknown",
            "secondary": "",
            "primary_ratio": 0.0,
            "secondary_ratio": 0.0,
            "is_bicolor": False,
            "H": 0.0,
            "S": 0.0,
            "V": 0.0,
        }

    # quitar highlights extremos
    px = pixels[(pixels[:, 2] < 245)]
    if len(px) == 0:
        px = pixels

    h_med = float(np.median(px[:, 0]))
    s_med = float(np.percentile(px[:, 1], 40))
    v_med = float(np.percentile(px[:, 2], 75))

    # Clasificacion pixel a pixel para detectar bi-color
    labels = []
    for h, s, v in px:
        labels.append(_hsv_to_color_name(float(h), float(s), float(v)))

    counts = {}
    for name in labels:
        counts[name] = counts.get(name, 0) + 1

    # filtrar ruido muy chico
    total = max(len(labels), 1)
    filtered = {k: v for k, v in counts.items() if (v / total) >= 0.03}
    if not filtered:
        filtered = counts

    ranked = sorted(filtered.items(), key=lambda kv: kv[1], reverse=True)
    primary = ranked[0][0]
    primary_ratio = ranked[0][1] / total

    secondary = ""
    secondary_ratio = 0.0
    if len(ranked) > 1:
        secondary = ranked[1][0]
        secondary_ratio = ranked[1][1] / total

    # color robusto central para estabilizar mono-color
    central_label = _hsv_to_color_name(h_med, s_med, v_med)

    # bicolor cuando ambos colores tienen presencia real
    is_bicolor = (
        secondary != ""
        and secondary != primary
        and primary_ratio >= 0.30
        and secondary_ratio >= 0.18
    )

    # Si no hay evidencia clara de bicolor, priorizar label central (menos sensible a ruido)
    if not is_bicolor:
        if primary_ratio < 0.58 or primary == "unknown":
            if central_label != "unknown":
                primary = central_label

    return {
        "mode": "bicolor" if is_bicolor else "mono",
        "primary": primary,
        "secondary": secondary if is_bicolor else "",
        "primary_ratio": float(primary_ratio),
        "secondary_ratio": float(secondary_ratio if is_bicolor else 0.0),
        "is_bicolor": bool(is_bicolor),
        "H": h_med,
        "S": s_med,
        "V": v_med,
    }


def detect_bicolor_split(roi, cnt):
    hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)

    obj_mask = np.zeros(roi.shape[:2], dtype=np.uint8)
    cv2.drawContours(obj_mask, [cnt], -1, 255, -1)
    inner = cv2.erode(obj_mask, np.ones((5, 5), np.uint8), iterations=1)
    if cv2.countNonZero(inner) < 30:
        inner = obj_mask

    ys, xs = np.where(inner == 255)
    if len(xs) < 120:
        return {"is_bicolor": False, "primary": "", "secondary": "", "left_ratio": 0.0, "right_ratio": 0.0}

    bx, by, bw, bh = cv2.boundingRect(cnt)
    if bw >= bh:
        split_v = np.median(xs)
        left_sel = xs < split_v
        right_sel = xs >= split_v
    else:
        split_v = np.median(ys)
        left_sel = ys < split_v
        right_sel = ys >= split_v

    if np.count_nonzero(left_sel) < 40 or np.count_nonzero(right_sel) < 40:
        return {"is_bicolor": False, "primary": "", "secondary": "", "left_ratio": 0.0, "right_ratio": 0.0}

    def dominant_name(sel):
        side_pixels = hsv[ys[sel], xs[sel]]
        side_pixels = side_pixels[(side_pixels[:, 2] < 245)]
        if len(side_pixels) == 0:
            return "unknown", 0.0
        names = [_hsv_to_color_name(float(h), float(s), float(v)) for h, s, v in side_pixels]
        counts = {}
        for n in names:
            if n == "unknown":
                continue
            counts[n] = counts.get(n, 0) + 1
        if not counts:
            return "unknown", 0.0
        k, v = max(counts.items(), key=lambda kv: kv[1])
        return k, float(v / max(len(names), 1))

    left_name, left_ratio = dominant_name(left_sel)
    right_name, right_ratio = dominant_name(right_sel)

    is_bi = (
        left_name != "unknown"
        and right_name != "unknown"
        and left_name != right_name
        and left_ratio >= 0.30
        and right_ratio >= 0.30
    )

    if not is_bi:
        return {"is_bicolor": False, "primary": "", "secondary": "", "left_ratio": left_ratio, "right_ratio": right_ratio}

    return {
        "is_bicolor": True,
        "primary": left_name,
        "secondary": right_name,
        "left_ratio": left_ratio,
        "right_ratio": right_ratio,
    }


# =========================
# Cámara
# =========================
templates = load_templates()
if templates:
    print("Templates cargados:", list(templates.keys()))
else:
    print("Sin templates: usando reglas geométricas.")

cap = cv2.VideoCapture(CAM_INDEX)
if not cap.isOpened():
    cap = cv2.VideoCapture(CAM_INDEX, cv2.CAP_DSHOW)

print("q = salir")

while True:
    ok, frame = cap.read()
    if not ok:
        print("No se pudo leer la cámara.")
        break

    display = frame.copy()

    (x1, y1, x2, y2), roi = get_center_roi(frame)
    mask, paper_bgr, delta = build_mask_white_bg(roi)
    contours = get_valid_contours(mask)

    roi_vis = roi.copy()
    edge_ext_vis = np.zeros(roi.shape[:2], dtype=np.uint8)
    edge_int_vis = np.zeros(roi.shape[:2], dtype=np.uint8)

    cv2.rectangle(display, (x1, y1), (x2, y2), (0, 255, 255), 2)
    cv2.putText(display, "ROI", (x1, max(25, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    primary = select_primary_contour(contours, roi.shape)
    if primary is None:
        cv2.putText(roi_vis, "Sin pastilla detectada", (15, 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

    for cnt in ([primary] if primary is not None else []):
        cnt = smooth_contour(cnt)
        shape_label, shape_info = classify_shape(cnt, templates)
        color_info = classify_color(roi, cnt)

        # Bordes externos e internos
        edge_ext_vis = get_external_edges(roi.shape[:2], cnt)
        edge_int_vis, int_edge_density = get_internal_edges(roi, cnt)
        shape_info["int_edge_density"] = float(int_edge_density)

        # Pintar bordes para que sean claramente visibles en roi_result
        roi_vis[edge_ext_vis > 0] = (0, 255, 0)   # externo en verde
        roi_vis[edge_int_vis > 0] = (0, 0, 255)   # interno en rojo

        # correccion ligera para caplets redondeadas
        aspect = float(shape_info.get("aspect", 0.0))
        circularity = float(shape_info.get("circularity", 0.0))
        if shape_label == "pill" and aspect < 2.45 and circularity >= 0.62:
            shape_label = "caplet"
        if shape_label == "circle" and aspect > 1.20:
            shape_label = "caplet"

        # deteccion bicolor por mitades para pills/caplets alargadas
        if shape_label in ("pill", "caplet") and (aspect >= 1.45 or shape_info.get("int_edge_density", 0.0) >= 0.018):
            split_info = detect_bicolor_split(roi, cnt)
            if split_info["is_bicolor"]:
                shape_label = "pill"
                color_info["is_bicolor"] = True
                color_info["mode"] = "bicolor"
                color_info["primary"] = split_info["primary"]
                color_info["secondary"] = split_info["secondary"]
                color_info["secondary_ratio"] = min(split_info.get("left_ratio", 0.0), split_info.get("right_ratio", 0.0))

        # solo la forma pill puede ser bicolor
        if shape_label != "pill":
            color_info["is_bicolor"] = False
            color_info["mode"] = "mono"
            color_info["secondary"] = ""
            color_info["secondary_ratio"] = 0.0

        x, y, w, h = cv2.boundingRect(cnt)

        cv2.drawContours(roi_vis, [cnt], -1, (0, 255, 0), 2)
        cv2.rectangle(roi_vis, (x, y), (x + w, y + h), (255, 0, 0), 2)

        if color_info["is_bicolor"]:
            color_label = f"{color_info['primary']}/{color_info['secondary']}"
        else:
            color_label = color_info["primary"]

        size_label, size_ratio = estimate_size_label(cnt, roi.shape)

        label = f"{color_label} | {shape_label} | {size_label}"
        cv2.putText(
            roi_vis, label, (x, max(20, y - 8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 255, 0), 2
        )

        hsv_txt = f"H:{color_info.get('H', 0):.0f} S:{color_info.get('S', 0):.0f} V:{color_info.get('V', 0):.0f}"
        cv2.putText(
            roi_vis, hsv_txt, (x, y + h + 18),
            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1
        )
        edge_txt = f"Eint:{shape_info.get('int_edge_density', 0.0):.3f}"
        cv2.putText(
            roi_vis, edge_txt, (x, y + h + 36),
            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1
        )
        size_txt = f"Tam:{size_label} A:{size_ratio:.3f}"
        cv2.putText(
            roi_vis, size_txt, (x, y + h + 54),
            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1
        )

    paper_box = np.full((40, 120, 3), paper_bgr, dtype=np.uint8)
    cv2.imshow("paper_ref", paper_box)
    cv2.imshow("camera", display)
    cv2.imshow("delta_white", delta)
    cv2.imshow("mask", mask)
    cv2.imshow("edge_ext", edge_ext_vis)
    cv2.imshow("edge_int", edge_int_vis)
    cv2.imshow("roi_result", roi_vis)

    key = cv2.waitKey(1) & 0xFF
    if key == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()


























Sin templates: usando reglas geométricas.
q = salir
